In [1]:
import os
import torch
import torch.nn.functional as F
import threading
import time
import nest_asyncio
import uvicorn
import ipywidgets as widgets
from IPython.display import display, clear_output
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from pyngrok import ngrok
from google.colab import drive

# --- 1. EMERGENCY CLEANUP (Prevents Port/Tunnel Crashes) ---
PORT = 8000
!fuser -k {PORT}/tcp || true
ngrok.kill()

# --- 2. CONFIG ---
NGROK_TOKEN = "3AwaRSWVBr5tOwMZQ1LpNijr1kZ_4pH2RQyoswEmU1bTurG2K"
ngrok.set_auth_token(NGROK_TOKEN)
nest_asyncio.apply()
model_lock = threading.Lock() # Prevents multiple requests from crashing the GPU

# --- 3. LOAD MODEL (With Memory Optimization) ---
drive.mount("/content/drive", force_remount=True)
SAVE_DIR = "/content/drive/MyDrive/PhishingClassifier"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading model safely...")
model = AutoModelForSequenceClassification.from_pretrained(f"{SAVE_DIR}/model").to(device)
tokenizer = AutoTokenizer.from_pretrained(f"{SAVE_DIR}/model")
model.eval()

# --- 4. THREAD-SAFE PREDICT FUNCTION ---
def predict_safe(text: str, threshold: float = 0.5) -> dict:
    # Use the lock so only one request touches the GPU at a time
    with model_lock:
        inputs = tokenizer(text, max_length=256, padding="max_length", truncation=True, return_tensors="pt")
        ids = inputs["input_ids"].to(device)
        mask = inputs["attention_mask"].to(device)

        with torch.no_grad():
            logits = model(input_ids=ids, attention_mask=mask).logits
            probs = F.softmax(logits, dim=-1)[0]

        # Clear GPU cache after processing to prevent OOM
        if device.type == 'cuda':
            torch.cuda.empty_cache()

        p = round(probs[1].item(), 4)
        s = round(probs[0].item(), 4)

        return {
            "verdict": "PHISHING" if p >= threshold else "SAFE",
            "phishing_score": p,
            "confidence": round(max(s, p), 4)
        }

# --- 5. APP SETUP ---
app = FastAPI()

class EmailRequest(BaseModel):
    text: str
    threshold: float = 0.5

@app.post("/classify")
def classify(req: EmailRequest):
    try:
        return predict_safe(req.text, req.threshold)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# --- 6. BACKGROUND SERVER ---
def run_server():
    # 'workers=1' ensures we don't spawn multiple processes that steal RAM
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="error", workers=1)

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)
public_url = ngrok.connect(PORT).public_url

# --- 7. UI ---
clear_output()
print(f"🚀 API STABLE & LIVE\nURL: {public_url}\nDocs: {public_url}/docs")
print("-" * 55)
print("Note: If session still crashes, reduce MAX_LEN in the code.")

🚀 API STABLE & LIVE
URL: https://rootless-nonexhaustive-lloyd.ngrok-free.dev
Docs: https://rootless-nonexhaustive-lloyd.ngrok-free.dev/docs
-------------------------------------------------------
Note: If session still crashes, reduce MAX_LEN in the code.
